# RLHF on Kaggle — run once and walk away

**Fire-and-forget & GPU-adaptive.** Accelerator = **GPU**, Internet = **On**, then **Save & Run All (Commit)**
(headless: `scripts/run_on_kaggle.sh`, which forces a T4 via `--accelerator NvidiaTeslaT4`).
Auto-detects the card: **T4** → bf16; **P100** → torch 2.4.1 + fp32 (Kaggle's base torch lost P100 support).

Runs SFT → reward model → PPO → eval and writes **`RESULTS.md`**.

In [ ]:
import torch, json
cap  = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
P100 = tuple(cap) == (6, 0)   # Pascal P100: Kaggle's base torch dropped its sm_60 kernels
json.dump({'p100': P100}, open('/tmp/gpu.json', 'w'))
print(f'GPU: {name}  cap={cap}  ->  ' + ('P100 path: torch 2.4.1 + fp32' if P100 else 'T4/base path: bf16'))

In [ ]:
# transformers <5 always (5.x broke the Qwen2.5 fast tokenizer on Kaggle).
!pip install -q 'transformers>=4.44,<5' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12' pytest
import json, subprocess, sys
if json.load(open('/tmp/gpu.json'))['p100']:
    # P100: base torch has no sm_60 kernels. Install torch 2.4.1 (+matched vision/audio) and pin
    # transformers to that era (4.47 imports Qwen2 cleanly on 2.4.1; matched torchvision avoids
    # the 'operator torchvision::nms does not exist' import crash).
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<4.48'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.4.1','torchvision==0.19.1',
                    'torchaudio==2.4.1','--index-url','https://download.pytorch.org/whl/cu121'], check=True)
    print('installed torch 2.4.1 + matched torchvision/torchaudio for the P100')

In [ ]:
import os
REPO_URL = 'https://github.com/TheYellowDuck/RLHF-pipeline.git'
ATTACHED = '/kaggle/input/rlhf-pipeline'
if REPO_URL:
    !rm -rf /kaggle/working/rlhf-pipeline && git clone -q $REPO_URL /kaggle/working/rlhf-pipeline
elif os.path.exists(ATTACHED):
    !rm -rf /kaggle/working/rlhf-pipeline && cp -r $ATTACHED /kaggle/working/rlhf-pipeline
else:
    raise SystemExit('Set REPO_URL or attach the repo as a Kaggle Dataset.')
%cd /kaggle/working/rlhf-pipeline

In [ ]:
# Pre-flight: proves the pipeline is sound on this box (tiny model, ~1 min). Aborts the run if red.
!python scripts/smoke_test.py && python -m pytest tests/ -q

## Config — the only knobs (precision + batch auto-set per GPU)

In [ ]:
import json
P100 = json.load(open('/tmp/gpu.json'))['p100']
MODEL   = 'Qwen/Qwen2.5-0.5B'
PRESET  = 'full'                # 'fast' (~2 h)  or  'full'
DATASET = 'uf'                  # 'uf' = UltraFeedback (cleaner, higher RM acc) | 'hh' = Anthropic/hh-rlhf

if DATASET == 'uf':
    DATA, TRAIN_SPLIT, EVAL_SPLIT = 'HuggingFaceH4/ultrafeedback_binarized', 'train_prefs', 'test_prefs'
else:
    DATA, TRAIN_SPLIT, EVAL_SPLIT = 'Anthropic/hh-rlhf', 'train', 'test'

if PRESET == 'fast':
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 4000, 8000, 1024, 32
else:
    SFT_SAMPLES, RM_SAMPLES, PPO_EPISODES, MAX_NEW = 8000, 20000, 2048, 40

# Precision per GPU; P100 fp32 needs smaller RM/PPO batches than T4 bf16.
if P100:
    DTYPE, BF16, RM_BATCH, RM_GA, PPO_MINI = 'float32', 'false', 4, 2, 2
else:
    DTYPE, BF16, RM_BATCH, RM_GA, PPO_MINI = 'bfloat16', 'true', 8, 1, 4
# SFT batch is tiny on ANY card: Qwen2.5's ~152k vocab makes the CausalLM loss upcast logits to
# fp32 (~2.3 GB at batch 8) -> OOM even in bf16. Batch 2 x grad-accum 4 keeps the effective batch 8.
SFT_BATCH, SFT_GA = 2, 4
ROLLOUT, PPO_LR = 16, '5e-7'
print(f'GPU={"P100/fp32" if P100 else "T4/bf16"}  dtype={DTYPE}  sft={SFT_BATCH}x{SFT_GA}  rm={RM_BATCH}x{RM_GA}  ppo_mini={PPO_MINI}')
print(f'model={MODEL}  data={DATA}  sft={SFT_SAMPLES}  rm={RM_SAMPLES}  ppo_ep={PPO_EPISODES}')

## 1. SFT

In [ ]:
!python scripts/train_sft.py \
  -o model.name_or_path=$MODEL -o model.dtype=$DTYPE -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT -o data.eval_split=$EVAL_SPLIT \
  -o data.max_samples=$SFT_SAMPLES -o train.batch_size=$SFT_BATCH -o train.grad_accum=$SFT_GA -o train.bf16=$BF16 -o output_dir=checkpoints/sft

## 2. Reward model  (scalar head; init from SFT + label-smoothed loss)

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=checkpoints/sft -o model.dtype=$DTYPE -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT -o data.eval_split=$EVAL_SPLIT \
  -o data.max_samples=$RM_SAMPLES -o data.max_eval_samples=500 \
  -o train.batch_size=$RM_BATCH -o train.grad_accum=$RM_GA -o train.bf16=$BF16 -o output_dir=checkpoints/reward_model

## 3. PPO (RL against the reward model)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=checkpoints/sft -o policy.dtype=$DTYPE \
  -o reward_model.name_or_path=checkpoints/reward_model -o reward_model.dtype=$DTYPE \
  -o data.name=$DATA -o data.train_split=$TRAIN_SPLIT \
  -o ppo.total_episodes=$PPO_EPISODES -o ppo.rollout_batch_size=$ROLLOUT -o ppo.mini_batch_size=$PPO_MINI \
  -o ppo.lr=$PPO_LR -o ppo.normalize_rewards=true \
  -o generation.max_new_tokens=$MAX_NEW -o data.max_prompt_length=160

## 4. Results → RESULTS.md

In [ ]:
import subprocess
def run(cmd):
    print('$', cmd, flush=True)
    o = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(o.stdout)
    if o.returncode: print(o.stderr[-1500:])
    return o.stdout

rm  = run(f'python scripts/evaluate.py rm-accuracy --reward-model checkpoints/reward_model --data {DATA} --split {EVAL_SPLIT} --max-samples 500')
win = run(f'python scripts/evaluate.py score-policy --policy checkpoints/ppo --reward-model checkpoints/reward_model --compare checkpoints/sft --data {DATA} --split {TRAIN_SPLIT} --num 100 --max-new-tokens {MAX_NEW}')
md = (f'# RLHF run results\n\n- model: `{MODEL}`\n- dataset: `{DATA}`\n- preset: `{PRESET}`\n\n'
      f'## Reward-model accuracy (held-out)\n```\n{rm}\n```\n\n'
      f'## PPO vs SFT — reward-model win-rate + sample completions\n```\n{win}\n```\n')
open('/kaggle/working/RESULTS.md','w').write(md)
print('\nSaved -> /kaggle/working/RESULTS.md')

### When it finishes
Output → **`RESULTS.md`** + `rlhf-pipeline/checkpoints/`. RM accuracy ~0.66–0.72 is realistic at this scale.
Still OOM on PPO? add `-o policy.use_lora=true`. PPO can resume after preemption with `--resume`.